In [0]:
%run ../../../utils/pipeline_utils


In [0]:
# Databricks notebook source
# =============================================================================
# common/prime_dim_date.py
# Primes assist_dev.common.dim_date
#
# Source      : Generated (no Silver source required)
# Grain       : One row per calendar date, 2000-01-01 → 2035-12-31
# SCD Type    : 1 (full overwrite, no SCD2 cols on dim_date)
# Idempotent  : YES — TRUNCATE then INSERT
# Dependencies: None
# =============================================================================

# COMMAND ----------
# MAGIC %run ../utils/pipeline_utils
dbutils.widgets.text("run_id",   "", "Pipeline Run ID")
dbutils.widgets.text("job_name", "dp1_prime_full", "Job Name")

RUN_ID   = dbutils.widgets.get("run_id")   
#or "manual-" + get_spark_app_id()
JOB_NAME = dbutils.widgets.get("job_name")

TARGET   = gold("common", "dim_date")
TASK     = "prime_dim_date"

print(f"[{TASK}] target={TARGET} run_id={RUN_ID}")

# COMMAND ----------
# ── Step 1 : Audit start ──────────────────────────────────────────────────────
start_ts = audit_start(spark, RUN_ID, JOB_NAME, TASK, TARGET,
                        source_schema="generated", source_table="date_spine")

# COMMAND ----------
try:
    # ── Step 2 : TRUNCATE ────────────────────────────────────────────────────
    truncate_gold(spark, TARGET)

    # ── Step 3 : Generate date spine via Spark SQL ───────────────────────────
    # Federal fiscal year starts October 1.
    # FY2025 = Oct 1 2024 → Sep 30 2025 → fy = calendar_year + (1 if month >= 10 else 0)
    # Federal fiscal month 1 = October, 12 = September
    # Federal fiscal quarter:  Q1=Oct-Dec, Q2=Jan-Mar, Q3=Apr-Jun, Q4=Jul-Sep

    spark.sql(f"""
        INSERT INTO {TARGET}
        (
            date_sk, calendar_date, day_of_week, day_of_week_num,
            day_of_month, day_of_year, week_of_year,
            calendar_month, month_name, calendar_quarter, calendar_year,
            federal_fiscal_fy, federal_fiscal_qtr, federal_fiscal_month,
            is_weekend_flag, is_federal_holiday,
            is_last_day_of_month, is_last_day_of_fy,
            _gold_created_at, _gold_updated_at, _source_batch_id
        )
        WITH date_spine AS (
            SELECT EXPLODE(
                SEQUENCE(DATE '2000-01-01', DATE '2035-12-31', INTERVAL 1 DAY)
            ) AS calendar_date
        ),
        enriched AS (
            SELECT
                calendar_date,
                -- Surrogate key: YYYYMMDD integer
                CAST(DATE_FORMAT(calendar_date, 'yyyyMMdd') AS INT)     AS date_sk,
                DATE_FORMAT(calendar_date, 'EEEE')                      AS day_of_week,
                -- Databricks DAYOFWEEK: 1=Sunday … 7=Saturday → remap to 1=Monday
                CASE WHEN DAYOFWEEK(calendar_date) = 1 THEN 7
                     ELSE DAYOFWEEK(calendar_date) - 1 END               AS day_of_week_num,
                DAYOFMONTH(calendar_date)                                AS day_of_month,
                DAYOFYEAR(calendar_date)                                 AS day_of_year,
                WEEKOFYEAR(calendar_date)                                AS week_of_year,
                MONTH(calendar_date)                                     AS calendar_month,
                DATE_FORMAT(calendar_date, 'MMMM')                      AS month_name,
                QUARTER(calendar_date)                                   AS calendar_quarter,
                YEAR(calendar_date)                                      AS calendar_year,

                -- Federal fiscal year (Oct 1 start)
                CASE WHEN MONTH(calendar_date) >= 10
                     THEN YEAR(calendar_date) + 1
                     ELSE YEAR(calendar_date)
                END                                                      AS federal_fiscal_fy,

                -- Federal fiscal quarter (Q1=Oct–Dec, Q2=Jan–Mar, Q3=Apr–Jun, Q4=Jul–Sep)
                CASE
                    WHEN MONTH(calendar_date) IN (10, 11, 12) THEN 1
                    WHEN MONTH(calendar_date) IN (1,  2,  3)  THEN 2
                    WHEN MONTH(calendar_date) IN (4,  5,  6)  THEN 3
                    ELSE 4
                END                                                      AS federal_fiscal_qtr,

                -- Federal fiscal month (1=Oct … 12=Sep)
                CASE WHEN MONTH(calendar_date) >= 10
                     THEN MONTH(calendar_date) - 9
                     ELSE MONTH(calendar_date) + 3
                END                                                      AS federal_fiscal_month,

                -- Weekend flag
                DAYOFWEEK(calendar_date) IN (1, 7)                      AS is_weekend_flag,

                -- US federal holiday approximation (fixed-date rules only)
                -- Full observance rules (floating Mondays) are better maintained
                -- as a reference table; this covers the majority of cases.
                CASE WHEN
                    -- New Year's Day
                    (MONTH(calendar_date) = 1  AND DAYOFMONTH(calendar_date) = 1)
                    -- MLK Day (3rd Monday of January) — approximated here as FALSE;
                    -- use DP11 lu_ reference for precise floating holiday logic
                    OR (MONTH(calendar_date) = 7  AND DAYOFMONTH(calendar_date) = 4)   -- Independence Day
                    OR (MONTH(calendar_date) = 11 AND DAYOFMONTH(calendar_date) = 11)  -- Veterans Day
                    OR (MONTH(calendar_date) = 12 AND DAYOFMONTH(calendar_date) = 25)  -- Christmas
                    THEN TRUE ELSE FALSE
                END                                                      AS is_federal_holiday,

                -- Last day of calendar month
                calendar_date = LAST_DAY(calendar_date)                 AS is_last_day_of_month,

                -- Last day of federal fiscal year = September 30
                (MONTH(calendar_date) = 9 AND DAYOFMONTH(calendar_date) = 30)
                                                                         AS is_last_day_of_fy
            FROM date_spine
        )
        SELECT
            date_sk, calendar_date, day_of_week, day_of_week_num,
            day_of_month, day_of_year, week_of_year,
            calendar_month, month_name, calendar_quarter, calendar_year,
            federal_fiscal_fy, federal_fiscal_qtr, federal_fiscal_month,
            is_weekend_flag, is_federal_holiday,
            is_last_day_of_month, is_last_day_of_fy,
            CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), '{RUN_ID}'
        FROM enriched
        ORDER BY calendar_date
    """)

    # ── Step 4 : Verify ──────────────────────────────────────────────────────
    n = row_count(spark, TARGET)
    print(f"  [OK] Inserted {n:,} date rows (expected ~13,150 for 2000–2035)")

    # Quick sanity: FY2025 should start 2024-10-01
    check = spark.sql(f"""
        SELECT calendar_date, federal_fiscal_fy, federal_fiscal_qtr, federal_fiscal_month
        FROM {TARGET}
        WHERE calendar_date = DATE '2024-10-01'
    """).collect()
    assert check[0]["federal_fiscal_fy"]    == 2025, "FY check failed"
    assert check[0]["federal_fiscal_qtr"]   == 1,    "FQ check failed"
    assert check[0]["federal_fiscal_month"] == 1,    "FM check failed"
    print("  [ASSERT] Federal fiscal calendar validated ✓")

    # ── Step 5 : Audit success ───────────────────────────────────────────────
    audit_success(spark, RUN_ID, TARGET, n, n, start_ts)

except Exception as e:
    audit_fail(spark, RUN_ID, TARGET, str(e), traceback.format_exc(), start_ts)
    raise

# COMMAND ----------
dbutils.notebook.exit("SUCCESS")
